In [ ]:
#all the necessary IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from torch.optim.lr_scheduler import LambdaLR
from torch.cuda.amp import GradScaler, autocast
from collections import OrderedDict, Counter
from itertools import chain

import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import math
import time
import os
import gc
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.model_selection import train_test_split


#CUDA Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
#Loading and spliting data into train and val set to get Bleu Score

with open('/kaggle/input/preprocessed-data/train.pkl', 'rb') as file:
    data = pickle.load(file)
with open('/kaggle/input/test-data/preprocessed_testEnglish-Bengali_ids.pkl', 'rb') as file:
    ids = pickle.load(file)
with open('/kaggle/input/test-data/preprocessed_testEnglish-Bengali_sentence.pkl', 'rb') as file:
    sent = pickle.load(file)

en_all_train = data["source_Bengali"]
de_all_train = data["target_Bengali"]
en_test_tokens = sent
ids_val = ids

print(f"Train EN: {len(en_all_train)}, Train DE: {len(de_all_train)}, Test: {len(en_test_tokens)}")

# Train/Val split
en_train_tokens, en_val_tokens, de_train_tokens, de_val_tokens = train_test_split(
    en_all_train, de_all_train, test_size=0.05, random_state=42
)

print(f"Train: {len(en_train_tokens)}, Val: {len(en_val_tokens)}, Test: {len(en_test_tokens)}")

In [ ]:
#Vocabulary building and Encoding data

# Special tokens
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
special_tokens = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

# Build vocabularies
def build_vocab(token_lists, min_freq=2):
    counter = Counter(chain.from_iterable(token_lists))
    vocab = [token for token, count in counter.items() if count >= min_freq]
    vocab = special_tokens + vocab
    word2index = {token: idx for idx, token in enumerate(vocab)}
    index2word = {idx: token for token, idx in word2index.items()}
    return word2index, index2word, vocab

en_word2index, en_index2word, en_vocab_list = build_vocab(en_train_tokens + en_val_tokens + en_test_tokens)
de_word2index, de_index2word, de_vocab_list = build_vocab(de_train_tokens + de_val_tokens)

PAD_IDX = en_word2index[PAD_TOKEN]
SOS_IDX = en_word2index[SOS_TOKEN]
EOS_IDX = en_word2index[EOS_TOKEN]
UNK_IDX = en_word2index[UNK_TOKEN]

print(f"EN vocab size: {len(en_word2index)}")
print(f"DE vocab size: {len(de_word2index)}")
print(f"PAD={PAD_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}, UNK={UNK_IDX}")

seq_length = 55

def encode_and_pad(word2index, tokens, max_length, unk_idx=UNK_IDX):
    """Encode tokens to indices and pad"""
    encoded = [word2index.get(token, unk_idx) for token in tokens]
    # Add SOS at start, EOS at end
    encoded = [SOS_IDX] + encoded + [EOS_IDX]
    
    # Pad or truncate to max_length
    if len(encoded) < max_length:
        encoded = encoded + [PAD_IDX] * (max_length - len(encoded))
    else:
        encoded = encoded[:max_length]
    
    return encoded

# Encode all data
print("Encoding training data...")
train_x_encoded = np.array([encode_and_pad(en_word2index, tokens, seq_length) for tokens in tqdm(en_train_tokens)])
train_y_encoded = np.array([encode_and_pad(de_word2index, tokens, seq_length) for tokens in tqdm(de_train_tokens)])

print("Encoding validation data...")
val_x_encoded = np.array([encode_and_pad(en_word2index, tokens, seq_length) for tokens in tqdm(en_val_tokens)])
val_y_encoded = np.array([encode_and_pad(de_word2index, tokens, seq_length) for tokens in tqdm(de_val_tokens)])

print("Encoding test data...")
test_x_encoded = np.array([encode_and_pad(en_word2index, tokens, seq_length) for tokens in tqdm(en_test_tokens)])

In [ ]:
#Dataloaders

train_batch_size = 32
inference_batch_size = 256

train_ds = TensorDataset(torch.from_numpy(train_x_encoded).long(), torch.from_numpy(train_y_encoded).long())
val_ds = TensorDataset(torch.from_numpy(val_x_encoded).long(), torch.from_numpy(val_y_encoded).long())
test_ds = TensorDataset(torch.from_numpy(test_x_encoded).long())

train_dl = DataLoader(train_ds, batch_size=train_batch_size, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=inference_batch_size, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=inference_batch_size, shuffle=False)

print(f"Train batches: {len(train_dl)}, Val batches: {len(val_dl)}, Test batches: {len(test_dl)}")

In [ ]:
#Defining model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.15, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size, nhead,
                 src_vocab_size, tgt_vocab_size, dim_feedforward, dropout=0.1):
        super(Seq2SeqTransformer, self).__init__()
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=F.gelu,
            batch_first=True,
            norm_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=emb_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=F.gelu,
            batch_first=True,
            norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=PAD_IDX)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=PAD_IDX)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)
        self.emb_size = emb_size

        # Initialize weights
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
                
        # Weight tying
        self.generator.weight = self.tgt_tok_emb.weight

    def forward(self, src, tgt, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src) * math.sqrt(self.emb_size))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt) * math.sqrt(self.emb_size))
        
        memory = self.transformer_encoder(src=src_emb, mask=None, src_key_padding_mask=src_padding_mask)
        output = self.transformer_decoder(tgt=tgt_emb, memory=memory, tgt_mask=tgt_mask,
                                         memory_mask=None, tgt_key_padding_mask=tgt_padding_mask,
                                         memory_key_padding_mask=memory_key_padding_mask)
        return self.generator(output)

    def encode(self, src, src_padding_mask):
        src_emb = self.positional_encoding(self.src_tok_emb(src) * math.sqrt(self.emb_size))
        return self.transformer_encoder(src_emb, None, src_padding_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_padding_mask, memory_key_padding_mask):
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt) * math.sqrt(self.emb_size))
        return self.transformer_decoder(tgt_emb, memory, tgt_mask, None, tgt_padding_mask, memory_key_padding_mask)

def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

print("Model architecture defined.")

In [ ]:
# Testing and Decoding Functions

def batch_greedy_decode(model, src, max_len, start_symbol):
    """Batched greedy decoding"""
    model.eval()
    src = src.to(device)
    batch_size = src.shape[0]
    src_padding_mask = (src == PAD_IDX).to(device)
    
    with torch.no_grad():
        memory = model.encode(src, src_padding_mask)
    
    ys = torch.ones(batch_size, 1).fill_(start_symbol).type(torch.long).to(device)
    ended = torch.zeros(batch_size).bool().to(device)
    
    for i in range(max_len - 1):
        with torch.no_grad():
            tgt_seq_len = ys.shape[1]
            tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
            tgt_padding_mask = (ys == PAD_IDX).to(device)
            out = model.decode(ys, memory, tgt_mask, tgt_padding_mask, src_padding_mask)
            prob = model.generator(out[:, -1])
        
        _, next_word = torch.max(prob, dim=1)
        next_word_col = next_word.unsqueeze(1)
        next_word_col[ended] = PAD_IDX
        ys = torch.cat([ys, next_word_col], dim=1)
        ended = ended | (next_word == EOS_IDX)
        if ended.all():
            break
    
    return ys[:, 1:]

def calculate_bleu(model, val_ds, num_sentences=100):
    """Calculate BLEU score"""
    model.eval()
    references = []
    candidates = []
    chencherry = SmoothingFunction().method1
    
    for idx in tqdm(range(min(num_sentences, len(val_ds))), desc="Calculating BLEU"):
        src_tensor, tgt_tensor = val_ds[idx]
        
        # Reference: remove SOS, EOS, PAD
        ref_indices = tgt_tensor.numpy()
        ref_tokens = [de_index2word.get(i, UNK_TOKEN) for i in ref_indices 
                      if i not in (PAD_IDX, SOS_IDX, EOS_IDX)]
        references.append([ref_tokens])
        
        # Prediction
        input_tensor = src_tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            pred_batch = batch_greedy_decode(model, input_tensor, seq_length, SOS_IDX)
        
        pred_indices = pred_batch[0].cpu().numpy()
        pred_tokens = [de_index2word.get(i, UNK_TOKEN) for i in pred_indices 
                       if i not in (PAD_IDX, EOS_IDX)]
        candidates.append(pred_tokens)
    
    if len(candidates) == 0 or len(references) == 0:
        return 0.0
    
    bleu_score = corpus_bleu(references, candidates, smoothing_function=chencherry)
    return bleu_score

In [ ]:
#Training loop along with prediction and selecting best model based on highes Bleu Score obtained per Epoch

EMB_SIZE = 512
NHEAD = 8
HID_DIM = 1024
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
DROPOUT = 0.15
EPOCHS = 15

CHECK = '/kaggle/working/training_checkpointB.pth'
BEST_MODEL_PATH = '/kaggle/working/best_modelB.pth'

model = Seq2SeqTransformer(
    NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE, NHEAD,
    len(en_word2index), len(de_word2index), HID_DIM, dropout=DROPOUT).to(device)

model = torch.compile(model)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)

# Warmup learning rate schedule
D_MODEL = EMB_SIZE
WARMUP_STEPS = 4000

def lr_lambda(step):
    step_num = step + 1
    arg1 = step_num ** -0.5
    arg2 = step_num * (WARMUP_STEPS ** -1.5)
    return (D_MODEL ** -0.5) * min(arg1, arg2)

scheduler = LambdaLR(optimizer, lr_lambda)

# Load checkpoint if exists
start_epoch = 1
best_bleu_score = 0.0
train_losses = []
global_step = 0

if os.path.exists(CHECK):
    print(f"\nLoading checkpoint from {CHECK}")
    checkpoint = torch.load(CHECK, map_location=device)
    model_state = checkpoint['model_state_dict']
    new_state_dict = OrderedDict()
    for k, v in model_state.items():
        name = k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k
        new_state_dict[name] = v
    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(new_state_dict)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_bleu_score = checkpoint['best_bleu_score']
    train_losses = checkpoint['train_losses']
    global_step = checkpoint['global_step']
    print(f"Resuming from epoch {start_epoch}")
    print(f"Best BLEU so far: {best_bleu_score:}")
else:
    print("Starting fresh training.")

scaler = GradScaler()

print(f"\nTraining from epoch {start_epoch} to {EPOCHS}...\n")

for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.time()
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for i, (src, tgt) in enumerate(progress_bar):
        src, tgt = src.to(device), tgt.to(device)
        trg_input, trg_output = tgt[:, :-1], tgt[:, 1:]
        
        tgt_seq_len = trg_input.shape[1]
        tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
        src_padding_mask = (src == PAD_IDX)
        tgt_padding_mask = (trg_input == PAD_IDX)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(src, trg_input, tgt_mask, src_padding_mask, tgt_padding_mask, src_padding_mask)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), trg_output.reshape(-1).long())
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        global_step += 1
        
        total_loss += loss.item()
        current_lr = scheduler.get_last_lr()[0]
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.6e}")
    
    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
    avg_epoch_loss = total_loss / len(train_dl)
    train_losses.append(avg_epoch_loss)
    
    print(f"Epoch {epoch}/{EPOCHS} Summary")
    print(f"Time: {epoch_mins:.0f}m {epoch_secs:.0f}s")
    print(f"Avg Loss: {avg_epoch_loss:.4f}")
    print(f"LR: {scheduler.get_last_lr()[0]:.6e}")
    
    # Evaluate BLEU
    print("\nEvaluating BLEU...")
    current_bleu_score = calculate_bleu(model, val_ds, num_sentences=40)
    print(f"BLEU Score: {current_bleu_score:}")
    
    # Save best model
    if current_bleu_score > best_bleu_score:
        best_bleu_score = current_bleu_score
        torch.save(getattr(model, '_orig_mod', model).state_dict(), BEST_MODEL_PATH)
        print(f" NEW BEST MODEL SAVED! BLEU: {best_bleu_score:}")
    
    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': getattr(model, '_orig_mod', model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_bleu_score': best_bleu_score,
        'train_losses': train_losses,
        'global_step': global_step,
    }, CHECK)

    # Generate predictions every 5 epochs or at last epoch
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"\nGenerating test predictions for epoch {epoch}...")
        inference_model = Seq2SeqTransformer(
            NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE, NHEAD,
            len(en_word2index), len(de_word2index), HID_DIM, dropout=DROPOUT
        ).to(device)
        best_state_dict = torch.load(BEST_MODEL_PATH, map_location=device)
        inference_model.load_state_dict(best_state_dict)
        inference_model = torch.compile(inference_model)
        inference_model.eval()
        
        val_outs = []
        for batch in tqdm(test_dl, desc="Generating predictions"):
            src = batch[0].to(device)
            with torch.no_grad():
                translation_indices_batch = batch_greedy_decode(inference_model, src, seq_length, SOS_IDX)
            
            for translation_indices in translation_indices_batch:
                pred_indices = translation_indices.cpu().numpy()
                pred_tokens = [de_index2word.get(i, UNK_TOKEN) for i in pred_indices if i not in (PAD_IDX, EOS_IDX)]
                pred_text = " ".join(pred_tokens)
                val_outs.append(pred_text)
        
        df_final = pd.DataFrame({
            'ID': ids_val,
            'Translation': val_outs
        })
        output_file = f'/kaggle/working/answers_epoch_{epoch}B.csv'
        df_final.to_csv(output_file, index=False)
        print(f" Predictions saved to {output_file}")
        
        del inference_model
        torch.cuda.empty_cache()
    
print(f"TRAINING COMPLETE!!")
print(f"Best BLEU Score: {best_bleu_score:}")
print(f"Best model saved at: {BEST_MODEL_PATH}")